<a href="https://colab.research.google.com/github/seifalaa1172003/BudgetBuddy/blob/master/ai_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pathlib, os
matches = list(pathlib.Path('/content/drive/MyDrive').rglob('train.py'))
project_dir = matches[0].parent
os.chdir(project_dir)
print("Now in:", os.getcwd())
!ls -la

Now in: /content/drive/MyDrive/ai model/java-vuln-multitask
total 63
drwx------ 4 root root  4096 Nov 22 18:33 .
drwx------ 3 root root  4096 Nov 22 18:32 ..
drwx------ 2 root root  4096 Nov 22 18:33 data
-rw------- 1 root root  3170 Nov 22 14:13 dataset.py
-rw------- 1 root root  9243 Nov 22 14:13 evaluate.py
-rw------- 1 root root  5836 Nov 22 14:13 model.py
-rw------- 1 root root  3255 Nov 22 14:13 predict.py
-rw------- 1 root root  8314 Nov 22 14:13 prepare_dataset.py
drwx------ 2 root root  4096 Nov 22 18:33 __pycache__
-rw------- 1 root root  3257 Nov 22 14:13 README.md
-rw------- 1 root root    81 Nov 22 14:13 requirements.txt
-rw------- 1 root root 11477 Nov 22 14:13 train.py


In [7]:
!nvidia-smi
!pip install -q "transformers>=4.40.0"

Sun Nov 22 18:38:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!python train.py --epochs 5 --batch-size 16 --grad-accum 2 --max-length 512 --num-workers 2

Using device: cuda (AMP=True)
config.json: 100% 691/691 [00:00<00:00, 3.23MB/s]
tokenizer_config.json: 100% 1.11k/1.11k [00:00<00:00, 4.26MB/s]
vocab.json: 100% 938k/938k [00:00<00:00, 33.7MB/s]
merges.txt: 100% 444k/444k [00:00<00:00, 89.9MB/s]
special_tokens_map.json: 100% 772/772 [00:00<00:00, 3.86MB/s]

pytorch_model.bin: downloading bytes:  93% 468M/504M [00:03<00:00, 316MB/s, 38.3MB/s  ]
pytorch_model.bin: reconstructing file:  26% 130M/504M [00:03<00:09, 39.1MB/s]
pytorch_model.bin: reconstructing file:  43% 215M/504M [00:03<00:04, 69.5MB/s, 12.0MB/s  ]
pytorch_model.bin: reconstructing file:  54% 274M/504M [00:03<00:02, 97.1MB/s, 19.5MB/s  ]
pytorch_model.bin: reconstructing file:  65% 328M/504M [00:03<00:01, 129MB/s, 24.7MB/s  ] 
pytorch_model.bin: downloading bytes: 100% 476M/476M [00:04<00:00, 108MB/s, 41.2MB/s  ]
pytorch_model.bin: reconstructing file: 100% 504M/504M [00:04<00:00, 114MB/s, 43.7MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 12745.14it/s]
Training:   0% 

In [9]:
import json, torch
from model import UniXcoderMultiTask
from predict import predict_code

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, tokenizer, cwe_label_map, config = UniXcoderMultiTask.load_pretrained(
    "checkpoints/unixcoder_multitask/best_model", device=device
)
id2cwe = {int(v): k for k, v in cwe_label_map.items()}
max_length = int(config.get("max_length", 512))

samples = {
    "1_SQL_Injection (expect CWE-89)": """
public ResultSet getUser(Connection conn, String username) throws SQLException {
    Statement stmt = conn.createStatement();
    String query = "SELECT * FROM users WHERE username = '" + username + "'";
    return stmt.executeQuery(query);
}
""",
    "2_Safe_SQL_PreparedStatement (expect BENIGN)": """
public ResultSet getUserSafe(Connection conn, String username) throws SQLException {
    PreparedStatement ps = conn.prepareStatement("SELECT * FROM users WHERE username = ?");
    ps.setString(1, username);
    return ps.executeQuery();
}
""",
    "3_Command_Injection (expect CWE-78)": """
public String pingHost(String host) throws IOException {
    Process p = Runtime.getRuntime().exec("ping -c 1 " + host);
    return new String(p.getInputStream().readAllBytes());
}
""",
    "4_Path_Traversal (expect CWE-22/23/36)": """
public byte[] readReport(String fileName) throws IOException {
    File file = new File("/var/app/reports/" + fileName);
    return Files.readAllBytes(file.toPath());
}
""",
    "5_Safe_Utility_Method (expect BENIGN)": """
public int sumPositiveNumbers(List<Integer> numbers) {
    if (numbers == null) return 0;
    int total = 0;
    for (int n : numbers) {
        if (n > 0) total += n;
    }
    return total;
}
""",
}

for name, code in samples.items():
    res = predict_code(model, tokenizer, id2cwe, code, device, max_length=max_length, threshold=0.5)
    print(f"=== {name} ===")
    print(json.dumps(res, indent=2))

=== 1_SQL_Injection (expect CWE-89) ===
{
  "is_vulnerable": true,
  "vulnerability_probability": 0.8994,
  "threshold": 0.5,
  "predicted_cwe": "CWE-89",
  "cwe_confidence": 0.9994,
  "top_k_cwes": [
    {
      "cwe_id": "CWE-89",
      "confidence": 0.9994
    },
    {
      "cwe_id": "CWE-79",
      "confidence": 0.0004
    },
    {
      "cwe_id": "CWE-78",
      "confidence": 0.0
    }
  ]
}
=== 2_Safe_SQL_PreparedStatement (expect BENIGN) ===
{
  "is_vulnerable": false,
  "vulnerability_probability": 0.1462,
  "threshold": 0.5,
  "predicted_cwe": null,
  "cwe_confidence": null,
  "top_k_cwes": [
    {
      "cwe_id": "CWE-89",
      "confidence": 0.9977
    },
    {
      "cwe_id": "CWE-79",
      "confidence": 0.0019
    },
    {
      "cwe_id": "CWE-78",
      "confidence": 0.0
    }
  ]
}
=== 3_Command_Injection (expect CWE-78) ===
{
  "is_vulnerable": false,
  "vulnerability_probability": 0.2829,
  "threshold": 0.5,
  "predicted_cwe": null,
  "cwe_confidence": null,
  "top_k

In [10]:
import shutil, zipfile, pathlib
from google.colab import files
# Clean folder for this upload
upload_dir = pathlib.Path("/content/uploaded_scan")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
uploaded = files.upload()  # Click "Choose Files" and select App.java (and optional pom.xml or .zip)
for filename, content in uploaded.items():
    dest = upload_dir / filename
    dest.write_bytes(content)
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(dest, "r") as zf:
            zf.extractall(upload_dir)
        dest.unlink()
!python scan_project.py \
  --target /content/uploaded_scan \
  --model-dir checkpoints/unixcoder_multitask/best_model \
  --threshold 0.5

Saving App.java to App.java
python3: can't open file '/content/drive/MyDrive/ai model/java-vuln-multitask/scan_project.py': [Errno 2] No such file or directory


In [11]:
from google.colab import files
files.upload()  # Select C:\Users\Admin\.gemini\antigravity\scratch\java-vuln-multitask\scan_project.py

Saving App.java to App (1).java


{'App (1).java': b'\r\nimport org.apache.commons.text.StringSubstitutor;\r\n\r\npublic class App {\r\n\r\n    public static void main(String[] args) {\r\n\r\n        if (args.length == 0) {\r\n            System.out.println("Please provide an input.");\r\n            return;\r\n        }\r\n\r\n        // Untrusted input\r\n        String userInput = args[0];\r\n\r\n        // Apache Commons Text\r\n        StringSubstitutor substitutor =\r\n                StringSubstitutor.createInterpolator();\r\n\r\n        // Process untrusted input\r\n        String result = substitutor.replace(userInput);\r\n\r\n        System.out.println(result);\r\n    }\r\n}\r\n'}

In [12]:
!python predict.py \
  --model-dir checkpoints/unixcoder_multitask/best_model \
  --file "App (1).java"

{
  "is_vulnerable": true,
  "vulnerability_probability": 0.999,
  "threshold": 0.5,
  "predicted_cwe": "CWE-79",
  "cwe_confidence": 0.3305,
  "top_k_cwes": [
    {
      "cwe_id": "CWE-79",
      "confidence": 0.3305
    },
    {
      "cwe_id": "CWE-74",
      "confidence": 0.2386
    },
    {
      "cwe_id": "CWE-134",
      "confidence": 0.1472
    }
  ]
}


In [13]:
from google.colab import files
files.upload()  # Select scan_project.py

Saving scan_project.py to scan_project.py


{'scan_project.py': b'"""Hybrid scanner for Java projects & libraries: OSV Maven dependency CVE scan + UniXcoderMultiTask code scan."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport bisect\nimport json\nimport os\nimport re\nimport urllib.error\nimport urllib.request\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\nfrom typing import Any\n\nSKIP_DIRS = {"target", "build", ".git", ".venv", "venv", "node_modules", "__pycache__"}\nNON_METHOD_NAMES = {\n    "if",\n    "for",\n    "while",\n    "switch",\n    "catch",\n    "synchronized",\n    "try",\n    "else",\n    "do",\n    "return",\n    "throw",\n    "new",\n    "super",\n    "this",\n    "assert",\n    "class",\n    "interface",\n    "enum",\n    "record",\n}\nTYPE_DECL_RE = re.compile(r"\\b(class|interface|enum|record)\\b")\nNEW_EXPR_RE = re.compile(r"\\bnew\\b")\nTHROWS_ONLY_RE = re.compile(\n    r"^(?:\\s*\\[\\s*\\])*\\s*(?:throws\\s+[A-Za-z0-9_.$<>\\s,\\[\\]]+)?$"\n)\nIDENT_BEFORE_PAREN_RE = re.c

In [19]:
import shutil, subprocess, zipfile, pathlib
from google.colab import files
scan_dir = pathlib.Path("/content/folder_to_scan")
if scan_dir.exists():
    shutil.rmtree(scan_dir)
scan_dir.mkdir(parents=True)
uploaded = files.upload()  # Select your .zip or .rar folder archive
for filename, content in uploaded.items():
    archive = scan_dir / filename
    archive.write_bytes(content)
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(scan_dir)
        archive.unlink()
    elif filename.lower().endswith(".rar"):
        subprocess.run(["apt-get", "install", "-qq", "unrar"], check=True)
        subprocess.run(["unrar", "x", "-o+", str(archive), str(scan_dir) + "/"], check=True)
        archive.unlink()
!python scan_project.py \
  --target /content/folder_to_scan \
  --model-dir checkpoints/unixcoder_multitask/best_model \
  --threshold 0.5 \
  --output-json scan_report.json

Saving vuln.java to vuln (1).java
=== Scan Target: /content/folder_to_scan ===
Java files found: 1 | Methods extracted: 0

--- Maven Dependency Scan (OSV.dev) | pom.xml files: 0 | queried: 0 ---
No known OSV vulnerabilities found in queried Maven dependencies.

--- ML Code Scan (UniXcoderMultiTask, threshold=0.5) | flagged: 0 / 0 methods ---
No methods above threshold; highest-scoring methods (showing 0):

Saved full JSON report to scan_report.json


In [20]:
import pathlib
p = pathlib.Path("scan_project.py")
text = p.read_text(encoding="utf-8")
old = '''            top_kind = block_stack[-1][0] if block_stack else "root"
            if top_kind == "root":
                block_stack.append(("type", sig_start, ""))
            elif top_kind == "type":
                outside = _strip_top_level_parens(raw_header)
                if TYPE_DECL_RE.search(outside):
                    block_stack.append(("type", sig_start, ""))
                else:
                    method_name = _detect_method_name(raw_header)
                    if method_name is not None:
                        block_stack.append(("method", sig_start, method_name))
                    else:
                        block_stack.append(("other", sig_start, ""))
            else:
                outside = _strip_top_level_parens(raw_header)
                if NEW_EXPR_RE.search(outside) or TYPE_DECL_RE.search(outside):
                    block_stack.append(("type", sig_start, ""))
                else:
                    block_stack.append(("other", sig_start, ""))'''
new = '''            top_kind = block_stack[-1][0] if block_stack else "root"
            outside = _strip_top_level_parens(raw_header)
            if top_kind in ("root", "type"):
                if TYPE_DECL_RE.search(outside):
                    block_stack.append(("type", sig_start, ""))
                else:
                    method_name = _detect_method_name(raw_header)
                    if method_name is not None:
                        block_stack.append(("method", sig_start, method_name))
                    else:
                        fallback_kind = "type" if top_kind == "root" else "other"
                        block_stack.append((fallback_kind, sig_start, ""))
            else:
                if NEW_EXPR_RE.search(outside) or TYPE_DECL_RE.search(outside):
                    block_stack.append(("type", sig_start, ""))
                else:
                    block_stack.append(("other", sig_start, ""))'''
assert old in text, "Already patched or unexpected file version"
p.write_text(text.replace(old, new, 1), encoding="utf-8")
print("Patched scan_project.py successfully!")

Patched scan_project.py successfully!


In [22]:
!python scan_project.py \
  --target /content/folder_to_scan \
  --model-dir checkpoints/unixcoder_multitask/best_model \
  --threshold 0.5 \
  --output-json scan_report.json

=== Scan Target: /content/folder_to_scan ===
Java files found: 1 | Methods extracted: 1

--- Maven Dependency Scan (OSV.dev) | pom.xml files: 0 | queried: 0 ---
No known OSV vulnerabilities found in queried Maven dependencies.

--- ML Code Scan (UniXcoderMultiTask, threshold=0.5) | flagged: 1 / 1 methods ---
Top flagged methods (showing 1):
  - vuln (1).java:1-26 [getUserByUsername] | prob=0.6070 | CWE-89 (1.00)

Saved full JSON report to scan_report.json


In [23]:
import shutil, subprocess, zipfile, pathlib
from google.colab import files
scan_dir = pathlib.Path("/content/folder_to_scan")
if scan_dir.exists():
    shutil.rmtree(scan_dir)
scan_dir.mkdir(parents=True)
uploaded = files.upload()  # Select your .zip or .rar folder archive
for filename, content in uploaded.items():
    archive = scan_dir / filename
    archive.write_bytes(content)
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(scan_dir)
        archive.unlink()
    elif filename.lower().endswith(".rar"):
        subprocess.run(["apt-get", "install", "-qq", "unrar"], check=True)
        subprocess.run(["unrar", "x", "-o+", str(archive), str(scan_dir) + "/"], check=True)
        archive.unlink()
!python scan_project.py \
  --target /content/folder_to_scan \
  --model-dir checkpoints/unixcoder_multitask/best_model \
  --threshold 0.5 \
  --output-json scan_report.json

Saving vuln.java to vuln (2).java
=== Scan Target: /content/folder_to_scan ===
Java files found: 1 | Methods extracted: 1

--- Maven Dependency Scan (OSV.dev) | pom.xml files: 0 | queried: 0 ---
No known OSV vulnerabilities found in queried Maven dependencies.

--- ML Code Scan (UniXcoderMultiTask, threshold=0.5) | flagged: 1 / 1 methods ---
Top flagged methods (showing 1):
  - vuln (2).java:1-26 [getUserByUsername] | prob=0.6070 | CWE-89 (1.00)

Saved full JSON report to scan_report.json
